In [ ]:
import os
import sys
sys.path.append(os.path.join(os.getcwd(), ".."))
from utils import find_nearest_gene_atac_peaks, preprocessing_rna, preprocessing_atac, preprocessing_prot
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import muon as mu
from tqdm import tqdm
tqdm.pandas()

Dataset has been downloaded from:
https://openproblems.bio/datasets/openproblems_neurips2021/bmmc_multiome/

In [ ]:
adata = sc.read_h5ad("./data/GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad")
adata

In [ ]:
random_state = np.random.RandomState()
indices = random_state.permutation(list(range(len(adata.obs))))
adata = adata[indices,:]

# RNA layer preprocessing

In [ ]:
rna = adata[:, adata.var["feature_types"] == "GEX"]

In [ ]:
rna.X = np.array(rna.layers["counts"].todense())

In [ ]:
rna.var["gene_stable_id"] = rna.var["gene_id"]

In [ ]:
rna = preprocessing_rna(rna, max_total_counts=15000)
rna

# ATAC layer preprocessing

In [ ]:
atac = adata[:, adata.var["feature_types"] == "ATAC"]

In [ ]:
atac.X = np.array(atac.layers["counts"].todense())

In [ ]:
atac.var["chr"] = list(atac.var.index.map(lambda x: x.split("-")[0]))
atac.var["start"] = list(atac.var.index.map(lambda x: x.split("-")[1]).astype("int"))
atac.var["end"] = list(atac.var.index.map(lambda x: x.split("-")[2]).astype("int"))
atac.var["gene_stable_id"] = list(atac.var.progress_apply(find_nearest_gene_atac_peaks, max_dist=10000, axis=1))

In [ ]:
atac = preprocessing_atac(atac)
atac

# Create MuData and save

In [ ]:
mdata = mu.MuData({"rna": rna, 
                   "atac": atac})

In [ ]:
mu.pp.intersect_obs(mdata)

In [ ]:
mdata.write("./data/neurips2021_multiome_BMMC.h5mu")

In [ ]:
mdata2 = mu.read("./data/neurips2021_multiome_BMMC.h5mu")
mdata2

In [ ]:
mdata2.mod["rna"]